# 06 - Baseline Models under the Corrected Protocol (P0)

**Purpose:** Fit all models under P0, storing MAE/RMSE/sMAPE/MASE and skill vs seasonal-naive, repeated across seeds and horizons.

**Inputs:** Feature matrices (notebook 04); validated P0 splitter (notebook 05).

**Outputs:** results/metrics/ P0 baseline metrics table.

**Runtime estimate:** 15-60 minutes depending on profile (fast vs full).

## Load experiment configuration and select profile (fast/full)

In [ ]:
import pandas as pd

from sdg_leakage_audit.utils.io import read_yaml
from sdg_leakage_audit.utils.paths import configs_dir, results_metrics_dir, to_project_relative

experiment_cfg_raw = read_yaml(configs_dir() / "experiment.yaml")
datasets_cfg = read_yaml(configs_dir() / "datasets.yaml")["datasets"]

ACTIVE_PROFILE = "full"  # Final-results run: all seeds, full data, full hyperparameter grid
profile = experiment_cfg_raw["profiles"][ACTIVE_PROFILE]
run_config = {
    "data_subsample_frac": profile["data_subsample_frac"],
    "model_param_grid": profile["model_param_grid"],
}
seeds = profile["seeds"]

print(f"Active profile: {ACTIVE_PROFILE!r} -> {profile['description']}")
print(f"run_config = {run_config}, seeds = {seeds}")


## Fit seasonal-naive and persistence baselines

In [ ]:
from sdg_leakage_audit.evaluation.runner import run_single_cell

results_rows = []

for dataset_id, cfg in datasets_cfg.items():
    for horizon in cfg["forecast_horizons"]:
        for seed in seeds:
            for model_name in ["seasonal_naive", "persistence"]:
                result = run_single_cell(dataset_id, "P0", model_name, horizon, seed, run_config)
                results_rows.append(result)

print(f"Baselines done: {len(results_rows)} cells (seasonal_naive + persistence).")
pd.DataFrame(results_rows).groupby(["dataset_id", "model_name"])[["mae", "smape"]].mean()


## Fit Ridge, RandomForest, LightGBM, MLP

In [ ]:
for dataset_id, cfg in datasets_cfg.items():
    for horizon in cfg["forecast_horizons"]:
        for seed in seeds:
            for model_name in ["ridge", "random_forest", "lightgbm", "mlp"]:
                result = run_single_cell(dataset_id, "P0", model_name, horizon, seed, run_config)
                results_rows.append(result)

print(f"ML models done: {len(results_rows)} cells total so far.")
pd.DataFrame(results_rows).groupby(["dataset_id", "model_name"])[["mae", "smape"]].mean().loc[
    pd.IndexSlice[:, ["ridge", "random_forest", "lightgbm", "mlp"]], :
]


## Fit small LSTM/GRU

In [ ]:
for dataset_id, cfg in datasets_cfg.items():
    for horizon in cfg["forecast_horizons"]:
        for seed in seeds:
            for model_name in ["lstm", "gru"]:
                result = run_single_cell(dataset_id, "P0", model_name, horizon, seed, run_config)
                results_rows.append(result)

print(f"All models done: {len(results_rows)} cells total.")
pd.DataFrame(results_rows).groupby(["dataset_id", "model_name"])[["mae", "smape"]].mean().loc[
    pd.IndexSlice[:, ["lstm", "gru"]], :
]


## Aggregate P0 metrics across seeds/horizons

In [ ]:
p0_metrics_df = pd.DataFrame(results_rows)

out_path = results_metrics_dir() / "p0_baseline_metrics.csv"
p0_metrics_df.to_csv(out_path, index=False)
print(f"Saved {len(p0_metrics_df)} rows -> {to_project_relative(out_path)}")

# Skill vs seasonal-naive, averaged across horizons/seeds, per dataset x model --
# the headline P0 baseline summary table for the rest of this project to build on.
skill_summary = (
    p0_metrics_df[p0_metrics_df["model_name"] != "seasonal_naive"]
    .groupby(["dataset_id", "model_name"])["skill_score"]
    .mean()
    .unstack("model_name")
    .round(3)
)
skill_summary


## Findings / what to carry forward

- All 8 models (seasonal-naive, persistence, Ridge, RandomForest, LightGBM, MLP, LSTM, GRU)
  were fit under the validated P0 protocol across all 5 datasets and each dataset's configured
  forecast horizons, using the `full` profile (all 5 seeds, full data, `params_full`
  hyperparameters) -- these are the final results, not a smoke test.
- Results saved to `results/metrics/p0_baseline_metrics.csv`; `skill_score` (fraction of
  seasonal-naive's error eliminated) is the headline comparison metric carried into notebook 07,
  where the same models are re-run under P1-P6 to measure leakage-driven inflation relative to
  these P0 numbers.
- Evaluation framing: "direct" horizon regression (each row's causal features predict the
  target `horizon` steps ahead of that row, evaluated independently per row) rather than a
  single-origin recursive multi-step forecast -- simpler to implement correctly and leakage-safe,
  at the cost of not modeling error accumulation across steps the way a recursive forecast would.
- Known approximation: the purge gap enforced by `chronological_split_with_gap` is `horizon`
  *positions* in the feature matrix after NaN-dropping, not strictly `horizon` *time steps*,
  wherever upstream missingness caused a few interior rows to be dropped. This is negligible
  for these datasets (missingness rates are low relative to horizon sizes) but is a known
  simplification worth stating explicitly rather than silently assuming exactness.
- LSTM/GRU operate on a short sequence built from each dataset's lag-feature columns (ordered by
  increasing lag), not a full sliding-window of raw observations -- a deliberate Phase-2 scope
  simplification; a true windowed-sequence recurrent model is a natural `full`-profile extension.
- Next (notebook 07): re-run this exact model/dataset/horizon grid under P1-P6 and compare against
  these P0 numbers to compute inflation.
